In [1]:
# CELL 0 — INSTALL
import subprocess, sys
def pip(*p): subprocess.run([sys.executable,'-m','pip','install','-q',*p], check=True)
pip('ultralytics==8.3.*','easyocr','opencv-python-headless','tqdm','basicsr','realesrgan')
print("✓ Done")

import os
import re
import cv2
import math
import json
import random
import shutil
import unicodedata
import yaml
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict
from ultralytics import YOLO
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 17.1 MB/s eta 0:00:00
✓ Done
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/qu

In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/tanvirmahtabtaneem/ocr-dataset/Bangla LPDB - A/Bangla_License_Plate"
)

WORK_ROOT = Path(
    "/kaggle/working/bangla_lpdb_yolo"
)

RUN_ROOT = Path(
    "/kaggle/working/bangla_lpdb_runs"
)

CLASS_FILE = DATASET_ROOT / "classes.txt"

SEED = 42

MODEL_NAME = "yolo11m.pt"

# Safer starting point for T4 16 GB x2.
# This is TOTAL batch size across both GPUs.
BATCH_SIZE = 8

IMG_SIZE = 960
EPOCHS = 80

# workers are used per GPU process in multi-GPU training
WORKERS = 2

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

MAKE_DEGRADED_COPY = True

In [3]:
# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# ============================================================
# 3. GPU CHECK
# ============================================================

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Enable GPU T4 x2 in Kaggle."
    )

if torch.cuda.device_count() >= 2:
    DEVICE = [0, 1]
    print("Training with BOTH GPUs:", DEVICE)
else:
    DEVICE = 0
    print("WARNING: Only one GPU is visible.")

GPU CHECK
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Training with BOTH GPUs: [0, 1]


In [5]:
# ============================================================
# 4. CHECK DATASET
# ============================================================

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset folder not found:\n{DATASET_ROOT}"
    )

if not CLASS_FILE.exists():
    raise FileNotFoundError(
        f"classes.txt not found:\n{CLASS_FILE}"
    )

In [6]:
# ============================================================
# 5. LOAD 102 CLASSES
# ============================================================

class_names = [
    x.strip()
    for x in CLASS_FILE.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()
    if x.strip()
]

print("\nNumber of classes:", len(class_names))

if len(class_names) != 102:
    print(
        "WARNING: expected 102 classes, "
        f"found {len(class_names)}."
    )

for i, name in enumerate(class_names):
    print(f"{i:3d} -> {name}")


Number of classes: 102
  0 -> 0
  1 -> 1
  2 -> 2
  3 -> 3
  4 -> 4
  5 -> 5
  6 -> 6
  7 -> 7
  8 -> 8
  9 -> 9
 10 -> Metro
 11 -> A
 12 -> Bha
 13 -> Cha
 14 -> Chha
 15 -> Da
 16 -> DA
 17 -> E
 18 -> Ga
 19 -> Gha
 20 -> Ha
 21 -> Ja
 22 -> Jha
 23 -> Ka
 24 -> Kha
 25 -> La
 26 -> Ma
 27 -> Na
 28 -> Pa
 29 -> Sa
 30 -> Sha
 31 -> Ta
 32 -> THA
 33 -> Tha
 34 -> U
 35 -> Bagerhat
 36 -> Bagura
 37 -> Bandarban
 38 -> Barguna
 39 -> Barisal
 40 -> Bhola
 41 -> Brahmanbaria
 42 -> Chandpur
 43 -> Chapainawabganj
 44 -> Chatto
 45 -> Chattogram
 46 -> Chuadanga
 47 -> Cox's Bazar
 48 -> Cumilla
 49 -> Dhaka
 50 -> Dinajpur
 51 -> Faridpur
 52 -> Feni
 53 -> Gaibandha
 54 -> Gazipur
 55 -> Gopalganj
 56 -> Habiganj
 57 -> Jamalpur
 58 -> Jessore
 59 -> Jhalokati
 60 -> Jhenaidah
 61 -> Joypurhat
 62 -> Khagrachari
 63 -> Khulna
 64 -> Kishoreganj
 65 -> Kurigram
 66 -> Kustia
 67 -> Lakshmipur
 68 -> Lalmonirhat
 69 -> Madaripur
 70 -> Magura
 71 -> Manikganj
 72 -> Meherpur
 73 -> 

In [7]:
# ============================================================
# 6. FIND IMAGE/LABEL PAIRS
# ============================================================

images = sorted(
    p
    for p in DATASET_ROOT.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
)

pairs = []

for image_path in images:

    label_path = image_path.with_suffix(".txt")

    if label_path.exists():
        pairs.append(
            (image_path, label_path)
        )

print("\nImages:", len(images))
print("Image/label pairs:", len(pairs))

if len(pairs) == 0:
    raise RuntimeError(
        "No image/label pairs were found."
    )


Images: 2668
Image/label pairs: 2668


In [8]:
# ============================================================
# 7. READ ALL CLASS IDS
# ============================================================

all_class_ids = []

for _, label_path in pairs:

    lines = label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()

    for line in lines:

        parts = line.split()

        if len(parts) != 5:
            continue

        try:
            class_id = int(float(parts[0]))
            all_class_ids.append(class_id)
        except ValueError:
            pass


if not all_class_ids:
    raise RuntimeError(
        "No valid YOLO class IDs were found."
    )

min_id = min(all_class_ids)
max_id = max(all_class_ids)
unique_ids = set(all_class_ids)

print("\nAnnotation class-ID range:")
print("Minimum:", min_id)
print("Maximum:", max_id)


Annotation class-ID range:
Minimum: 0
Maximum: 105


In [9]:
# ============================================================
# 8. HANDLE CLASS IDs
# ============================================================
#
# This dataset is already 0-based.
#
# classes.txt:
#   102 classes
#   valid IDs = 0..101
#
# Some annotation files contain:
#   102, 103, 104, 105
#
# These IDs do not exist in classes.txt.
# We will SKIP those invalid annotations.
# ============================================================

IS_ONE_BASED=False

VALID_CLASS_IDS=set(
    range(len(class_names))
)

EXTRA_CLASS_IDS=sorted(
    x for x in set(all_class_ids)
    if x not in VALID_CLASS_IDS
)

print("\nLabel convention:")
print("0-based labels detected.")
print(
    f"Valid class IDs: 0..{len(class_names)-1}"
)

if EXTRA_CLASS_IDS:

    print(
        "\nExtra class IDs found:",
        EXTRA_CLASS_IDS
    )

    print(
        "These invalid IDs will be skipped."
    )

else:

    print(
        "\nNo invalid class IDs found."
    )


Label convention:
0-based labels detected.
Valid class IDs: 0..101

Extra class IDs found: [102, 103, 105]
These invalid IDs will be skipped.


In [10]:
# ============================================================
# 9. NORMALIZE YOLO LABEL FILE
# ============================================================

def normalize_label_file(
    source_path,
    destination_path
):

    output_lines=[]

    lines=source_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()

    skipped=0

    for line in lines:

        parts=line.split()

        if len(parts)!=5:
            skipped+=1
            continue

        try:

            class_id=int(float(parts[0]))

            x=float(parts[1])
            y=float(parts[2])
            w=float(parts[3])
            h=float(parts[4])

        except ValueError:

            skipped+=1
            continue

        # ----------------------------------------------------
        # IMPORTANT:
        # Dataset is already 0-based.
        # Do NOT subtract 1.
        # ----------------------------------------------------

        if class_id not in VALID_CLASS_IDS:

            skipped+=1
            continue

        # ----------------------------------------------------
        # Check YOLO coordinates
        # ----------------------------------------------------

        if not all(
            0.0 <= value <= 1.0
            for value in [x,y,w,h]
        ):

            skipped+=1
            continue

        output_lines.append(
            f"{class_id} "
            f"{x:.6f} "
            f"{y:.6f} "
            f"{w:.6f} "
            f"{h:.6f}"
        )

    destination_path.write_text(
        "\n".join(output_lines),
        encoding="utf-8"
    )

    return skipped

In [11]:
# ============================================================
# 10. MOTION-BLUR / CCTV DEGRADATION
# ============================================================

def horizontal_motion_blur(
    image,
    kernel_size
):
    kernel = np.zeros(
        (kernel_size, kernel_size),
        dtype=np.float32
    )

    center = kernel_size // 2

    kernel[center, :] = (
        1.0 / kernel_size
    )

    return cv2.filter2D(
        image,
        -1,
        kernel
    )


def jpeg_compress(
    image,
    quality
):
    params = [
        cv2.IMWRITE_JPEG_QUALITY,
        quality
    ]

    success, encoded = cv2.imencode(
        ".jpg",
        image,
        params
    )

    if not success:
        return image

    return cv2.imdecode(
        encoded,
        cv2.IMREAD_COLOR
    )


def degrade_plate(image):

    output = image.copy()

    h, w = output.shape[:2]

    # --------------------------------------------------------
    # Motion blur
    # --------------------------------------------------------
    if random.random() < 0.85:

        kernel_size = random.choice(
            [5, 7, 9, 11, 15, 19]
        )

        output = horizontal_motion_blur(
            output,
            kernel_size
        )

    # --------------------------------------------------------
    # Low resolution
    # --------------------------------------------------------
    if random.random() < 0.75:

        scale = random.uniform(
            0.30,
            0.65
        )

        small_w = max(
            32,
            int(w * scale)
        )

        small_h = max(
            16,
            int(h * scale)
        )

        small = cv2.resize(
            output,
            (small_w, small_h),
            interpolation=cv2.INTER_AREA
        )

        output = cv2.resize(
            small,
            (w, h),
            interpolation=cv2.INTER_CUBIC
        )

    # --------------------------------------------------------
    # Brightness / contrast
    # --------------------------------------------------------
    if random.random() < 0.70:

        alpha = random.uniform(
            0.75,
            1.20
        )

        beta = random.randint(
            -25,
            25
        )

        output = cv2.convertScaleAbs(
            output,
            alpha=alpha,
            beta=beta
        )

    # --------------------------------------------------------
    # Sensor noise
    # --------------------------------------------------------
    if random.random() < 0.50:

        sigma = random.uniform(
            3.0,
            12.0
        )

        noise = np.random.normal(
            0,
            sigma,
            output.shape
        )

        output = np.clip(
            output.astype(np.float32)
            + noise,
            0,
            255
        ).astype(np.uint8)

    # --------------------------------------------------------
    # JPEG compression
    # --------------------------------------------------------
    if random.random() < 0.65:

        quality = random.randint(
            30,
            70
        )

        output = jpeg_compress(
            output,
            quality
        )

    return output

In [12]:
# ============================================================
# 11. TRAIN / VAL / TEST SPLIT
# ============================================================

random.shuffle(pairs)

total = len(pairs)

train_end = int(
    total * TRAIN_RATIO
)

val_end = (
    train_end
    + int(total * VAL_RATIO)
)

train_pairs = pairs[:train_end]
val_pairs = pairs[train_end:val_end]
test_pairs = pairs[val_end:]

print("\nSplit:")
print("Train:", len(train_pairs))
print("Val  :", len(val_pairs))
print("Test :", len(test_pairs))


Split:
Train: 2134
Val  : 266
Test : 268


In [13]:
# ============================================================
# 12. RESET WORKING DIRECTORY
# ============================================================

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

for split in [
    "train",
    "val",
    "test"
]:
    (
        WORK_ROOT
        / "images"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        WORK_ROOT
        / "labels"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

In [14]:
# ============================================================
# 13. COPY CLEAN DATA
# ============================================================

def copy_pair(
    image_path,
    label_path,
    split,
    suffix=""
):

    image = cv2.imread(
        str(image_path)
    )

    if image is None:
        raise RuntimeError(
            f"Cannot read image: {image_path}"
        )

    stem = (
        image_path.stem
        + suffix
    )

    output_image = (
        WORK_ROOT
        / "images"
        / split
        / f"{stem}.jpg"
    )

    output_label = (
        WORK_ROOT
        / "labels"
        / split
        / f"{stem}.txt"
    )

    cv2.imwrite(
        str(output_image),
        image,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            95
        ]
    )

    normalize_label_file(
        label_path,
        output_label
    )

    return image

In [15]:
# ============================================================
# 14. BUILD TRAINING DATASET
# ============================================================

print("\nBuilding training dataset...")

for image_path, label_path in train_pairs:

    image = copy_pair(
        image_path,
        label_path,
        "train"
    )

    # Additional degraded version
    if MAKE_DEGRADED_COPY:

        degraded = degrade_plate(
            image
        )

        stem = (
            image_path.stem
            + "_degraded"
        )

        output_image = (
            WORK_ROOT
            / "images"
            / "train"
            / f"{stem}.jpg"
        )

        output_label = (
            WORK_ROOT
            / "labels"
            / "train"
            / f"{stem}.txt"
        )

        cv2.imwrite(
            str(output_image),
            degraded,
            [
                cv2.IMWRITE_JPEG_QUALITY,
                95
            ]
        )

        normalize_label_file(
            label_path,
            output_label
        )


Building training dataset...


libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


In [16]:
# ============================================================
# 15. BUILD VALIDATION DATASET
# ============================================================

print("Building validation dataset...")

for image_path, label_path in val_pairs:

    copy_pair(
        image_path,
        label_path,
        "val"
    )

Building validation dataset...


In [17]:
# ============================================================
# 16. BUILD TEST DATASET
# ============================================================

print("Building test dataset...")

for image_path, label_path in test_pairs:

    copy_pair(
        image_path,
        label_path,
        "test"
    )

Building test dataset...


In [18]:
# ============================================================
# 17. SAVE CLASS MAP
# ============================================================

class_map = {
    str(i): name
    for i, name in enumerate(class_names)
}

(WORK_ROOT / "classes.json").write_text(
    json.dumps(
        class_map,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

1828

In [19]:
# ============================================================
# 18. CREATE YOLO data.yaml
# ============================================================

data_yaml = WORK_ROOT / "data.yaml"

yaml_data = {
    "path": str(WORK_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(class_names),
    "names": class_names
}

data_yaml.write_text(
    yaml.safe_dump(
        yaml_data,
        allow_unicode=True,
        sort_keys=False
    ),
    encoding="utf-8"
)

print("\nYOLO YAML:")
print(data_yaml)


YOLO YAML:
/kaggle/working/bangla_lpdb_yolo/data.yaml


In [20]:
# ============================================================
# 19. FINAL DATASET CHECK
# ============================================================

generated_labels = list(
    (WORK_ROOT / "labels").rglob("*.txt")
)

invalid_count = 0

for label_path in generated_labels:

    lines = label_path.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()

    for line in lines:

        parts = line.split()

        if len(parts) != 5:
            invalid_count += 1
            continue

        class_id = int(
            float(parts[0])
        )

        if not (
            0 <= class_id < len(class_names)
        ):
            invalid_count += 1

print("\nGenerated label files:", len(generated_labels))
print("Invalid label entries:", invalid_count)

if invalid_count != 0:
    raise RuntimeError(
        "Generated dataset still contains invalid labels."
    )

print("Dataset validation PASSED.")


Generated label files: 4802
Invalid label entries: 0
Dataset validation PASSED.


In [21]:
# ============================================================
# 20. TRAIN
# ============================================================

print("\n" + "=" * 70)
print("STARTING YOLO TRAINING")
print("=" * 70)

print("Model:", MODEL_NAME)
print("Image size:", IMG_SIZE)
print("Total batch:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Device:", DEVICE)


def train():

    model = YOLO(
        MODEL_NAME
    )

    model.train(

        data=str(data_yaml),

        epochs=EPOCHS,

        imgsz=IMG_SIZE,

        batch=BATCH_SIZE,

        device=DEVICE,

        workers=WORKERS,

        pretrained=True,

        optimizer="AdamW",

        lr0=0.001,

        lrf=0.01,

        weight_decay=0.0005,

        amp=True,

        cache=False,

        # Do not use mosaic/mixup for plate tokens.
        # They can distort or merge small characters.
        mosaic=0.0,
        mixup=0.0,

        # Keep geometry unchanged.
        degrees=0.0,
        translate=0.0,
        shear=0.0,
        perspective=0.0,

        # Small scale variation.
        scale=0.15,

        # Plates should not be flipped.
        fliplr=0.0,
        flipud=0.0,

        # Small lighting variation.
        hsv_h=0.0,
        hsv_s=0.10,
        hsv_v=0.20,

        patience=15,

        project=str(
            RUN_ROOT
        ),

        name="bangla_lpdb_yolo11m_t4x2",

        exist_ok=True,

        save=True,

        save_period=10,

        plots=True,

        verbose=True
    )


STARTING YOLO TRAINING
Model: yolo11m.pt
Image size: 960
Total batch: 8
Epochs: 80
Device: [0, 1]


In [22]:
# ============================================================
# 21. MAIN
# ============================================================

if __name__ == "__main__":

    train()

    print("\n" + "=" * 70)
    print("TRAINING FINISHED")
    print("=" * 70)

    best = (
        RUN_ROOT
        / "bangla_lpdb_yolo11m_t4x2"
        / "weights"
        / "best.pt"
    )

    print("Best model:")
    print(best)
    print("Exists:", best.exists())

New https://pypi.org/project/ultralytics/8.4.156 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/bangla_lpdb_yolo/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.1, hsv_v=0.2, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo1